# Pipeline de Produção - Anti-Money Laundering

Este notebook demonstra como usar o pipeline refatorado de detecção de lavagem de dinheiro.

**Características do Pipeline:**
- ✅ Zero Data Leakage (fit apenas em treino)
- ✅ Caminhos relativos (pathlib)
- ✅ Balanceamento seguro (RUS)
- ✅ Feature engineering consolidado
- ✅ Reprodutibilidade garantida

**Autor:** TCC - Anti-Money Laundering Detection  
**Data:** Janeiro 2026

## 1. Imports e Configuração

In [ ]:
import sys
from pathlib import Path

# Adicionar source ao path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from loguru import logger

# Módulos do projeto
from source.config import PROCESSED_DATA_DIR, MODELS_DIR, EXTERNAL_DATA_DIR, PROJ_ROOT
from source.preprocessing import AMLPreprocessor
from source.modeling.train_pipeline import AMLModelTrainer, load_data

# Configurações de visualização
plt.style.use('default')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)

logger.info(f"Diretório do projeto: {PROJ_ROOT}")
logger.info(f"Dados processados: {PROCESSED_DATA_DIR}")
logger.info(f"Modelos: {MODELS_DIR}")

## 2. Preparação de Dados (se necessário)

Se você ainda não executou `source/dataset.py`, pode fazê-lo diretamente:

In [ ]:
# Verificar se dados processados existem
treino_path = PROCESSED_DATA_DIR / 'df_treino.csv'
oot_path = PROCESSED_DATA_DIR / 'df_oot.csv'

if not treino_path.exists() or not oot_path.exists():
    logger.warning("Dados processados não encontrados. Execute source/dataset.py primeiro.")
    logger.info("Exemplo: python source/dataset.py")
else:
    logger.success("Dados processados encontrados!")
    logger.info(f"  - {treino_path.relative_to(PROJ_ROOT)}")
    logger.info(f"  - {oot_path.relative_to(PROJ_ROOT)}")

## 3. Carregar Dados

In [ ]:
# Carregar dados usando a função do módulo
X_train_raw, y_train, X_oot_raw, y_oot = load_data(PROCESSED_DATA_DIR)

logger.success("Dados carregados com sucesso!")
logger.info(f"  X_train: {X_train_raw.shape}")
logger.info(f"  y_train: {y_train.shape}")
logger.info(f"  X_oot:   {X_oot_raw.shape}")
logger.info(f"  y_oot:   {y_oot.shape}")

# Visualizar primeiras linhas
display(X_train_raw.head())

## 4. Configurar Preprocessador

O preprocessador aplica todas as transformações de forma segura (fit em treino, transform em OOT).

In [ ]:
# Configurar preprocessador
preprocessor = AMLPreprocessor(
    target_col='Is Laundering',
    onehot_cols=['Payment Format'],
    target_encoding_cols=['Receiving Currency', 'Payment Currency'],
    frequency_cols=[
        'Timestamp', 'From Account', 'To Account',
        'From Bank Name', 'Account Number', 'From Entity ID', 'From Entity Name',
        'To Bank Name', 'Account Number_To', 'To Entity ID', 'To Entity Name'
    ],
    transform_cols={
        'Amount Paid': 'yeojohnson',
        'Amount Received': 'yeojohnson',
        'Bank ID': 'yeojohnson',
        'Bank ID_To': 'yeojohnson',
        'From Bank': 'yeojohnson',
        'To Bank': 'yeojohnson'
    }
)

logger.info("Preprocessador configurado!")

## 5. Fit do Preprocessador (APENAS no Treino!)

**CRÍTICO**: O fit ocorre APENAS nos dados de treino para evitar data leakage.

In [ ]:
# Fit APENAS no treino
logger.info("Ajustando preprocessador no conjunto de TREINO...")
preprocessor.fit(X_train_raw, y_train)

logger.success(f"Preprocessador ajustado!")
logger.info(f"  Features geradas: {len(preprocessor.feature_names_)}")
logger.info(f"\nPrimeiras 20 features:")
for i, feat in enumerate(preprocessor.feature_names_[:20], 1):
    print(f"  {i:2d}. {feat}")

## 6. Transform em Treino e OOT

Agora aplicamos as transformações aprendidas do treino em ambos os conjuntos.

In [ ]:
# Transform em treino e OOT
logger.info("Transformando dados...")
X_train = preprocessor.transform(X_train_raw)
X_oot = preprocessor.transform(X_oot_raw)

logger.success("Transformação concluída!")
logger.info(f"  X_train: {X_train.shape}")
logger.info(f"  X_oot:   {X_oot.shape}")

# Verificar consistência de features
assert X_train.shape[1] == X_oot.shape[1], "Número de features deve ser igual!"
logger.success("✓ Número de features consistente entre treino e OOT")

# Visualizar dados transformados
display(X_train.head())

## 7. Análise de Distribuição de Classes

In [ ]:
# Visualizar distribuição
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Treino
y_train.value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'])
axes[0].set_title('Distribuição - Treino', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Classe')
axes[0].set_ylabel('Frequência')
axes[0].set_xticklabels(['Legítima', 'Fraude'], rotation=0)

# OOT
y_oot.value_counts().plot(kind='bar', ax=axes[1], color=['steelblue', 'coral'])
axes[1].set_title('Distribuição - OOT', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Classe')
axes[1].set_ylabel('Frequência')
axes[1].set_xticklabels(['Legítima', 'Fraude'], rotation=0)

plt.tight_layout()
plt.show()

# Estatísticas
logger.info(f"Distribuição Treino: {y_train.value_counts().to_dict()}")
logger.info(f"Distribuição OOT:    {y_oot.value_counts().to_dict()}")
logger.info(f"Taxa de Fraude Treino: {(y_train.sum() / len(y_train)) * 100:.2f}%")
logger.info(f"Taxa de Fraude OOT:    {(y_oot.sum() / len(y_oot)) * 100:.2f}%")

## 8. Treinamento de Modelos com RUS

Vamos treinar um modelo rápido (Logistic Regression) para demonstrar o pipeline completo.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Criar treinador
trainer = AMLModelTrainer(
    preprocessor=preprocessor,
    target_col='Is Laundering',
    random_state=42
)

# Treinar apenas Logistic Regression (rápido)
logger.info("Treinando Logistic Regression com RUS...")

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

pipeline = trainer.train_model(
    'Logistic Regression',
    lr_model,
    X_train,
    y_train,
    use_rus=True  # Aplica Random Under Sampling
)

logger.success("Modelo treinado!")

## 9. Avaliação do Modelo

In [ ]:
# Avaliar em treino
train_metrics = trainer.evaluate_model(
    'Logistic Regression',
    pipeline,
    X_train,
    y_train,
    'train'
)

# Avaliar em OOT
oot_metrics = trainer.evaluate_model(
    'Logistic Regression',
    pipeline,
    X_oot,
    y_oot,
    'oot'
)

# Comparar métricas
comparison = pd.DataFrame([train_metrics, oot_metrics])
comparison = comparison[['dataset', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']]

display(comparison)

## 10. Visualização de Resultados

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve, auc

# Predições
y_train_pred = pipeline.predict(X_train)
y_oot_pred = pipeline.predict(X_oot)
y_oot_proba = pipeline.predict_proba(X_oot)[:, 1]

# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Treino
cm_train = confusion_matrix(y_train, y_train_pred)
sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Matriz de Confusão - Treino', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predito')
axes[0].set_ylabel('Real')

# OOT
cm_oot = confusion_matrix(y_oot, y_oot_pred)
sns.heatmap(cm_oot, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title('Matriz de Confusão - OOT', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predito')
axes[1].set_ylabel('Real')

plt.tight_layout()
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(y_oot, y_oot_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taxa de Falsos Positivos', fontsize=11)
plt.ylabel('Taxa de Verdadeiros Positivos', fontsize=11)
plt.title('Curva ROC - OOT', fontsize=13, fontweight='bold')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

## 11. Salvar Preprocessador e Modelo

In [ ]:
# Salvar preprocessador
preprocessor_path = MODELS_DIR / 'preprocessor_demo.pkl'
preprocessor.save(preprocessor_path)

# Salvar modelo
import joblib
model_path = MODELS_DIR / 'logistic_regression_demo.pkl'
joblib.dump(pipeline, model_path)

logger.success("Artefatos salvos!")
logger.info(f"  - {preprocessor_path.relative_to(PROJ_ROOT)}")
logger.info(f"  - {model_path.relative_to(PROJ_ROOT)}")

## 12. Exemplo de Inferência (Carregando Modelo Salvo)

In [ ]:
# Simular ambiente de produção: carregar artefatos salvos
loaded_preprocessor = AMLPreprocessor.load(preprocessor_path)
loaded_model = joblib.load(model_path)

logger.info("Artefatos carregados da produção!")

# Pegar uma amostra do OOT para teste
sample = X_oot_raw.sample(5, random_state=42)
sample_y = y_oot.loc[sample.index]

# Transformar usando preprocessador carregado
sample_transformed = loaded_preprocessor.transform(sample)

# Predizer usando modelo carregado
predictions = loaded_model.predict(sample_transformed)
probabilities = loaded_model.predict_proba(sample_transformed)[:, 1]

# Mostrar resultados
results = pd.DataFrame({
    'Real': sample_y.values,
    'Predito': predictions,
    'Probabilidade_Fraude': probabilities
})

logger.info("\nResultados da Inferência:")
display(results)

## 13. Conclusão

Este notebook demonstrou:

✅ **Carregamento de dados** usando caminhos relativos  
✅ **Preprocessamento seguro** (fit em treino, transform em OOT)  
✅ **Balanceamento com RUS** aplicado apenas no treino  
✅ **Treinamento e avaliação** em conjuntos separados  
✅ **Persistência** de preprocessador e modelo  
✅ **Inferência** usando artefatos salvos

**Próximos Passos:**
1. Execute `python source/modeling/train_pipeline.py` para treinar todos os modelos
2. Compare resultados em `data/processed/model_results_oot.csv`
3. Selecione o melhor modelo para produção
4. Implemente API de inferência

**Lembre-se:**
- Sempre use os artefatos salvos (preprocessor + model) para inferência
- Nunca re-treine o preprocessador em dados de produção
- Monitore drift de dados em produção